# Attention Head Composition

Analyze how attention heads compose across layers: Q-composition, K-composition,
V-composition, virtual attention patterns, and composition path tracing.

Reference: Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"

## Why This Matters

Composition attention measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_composition import (
    qk_composition_scores,
    v_composition_scores,
    composition_path_tracing,
    virtual_attention_patterns,
    full_composition_matrix,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print("Model ready")

## Q/K Composition Scores

Q-composition measures how much an earlier head's OV output contributes to a later
head's query. K-composition measures the same for keys. High scores indicate the
later head uses information written by the earlier head to decide what to attend to.

In [ ]:
result = qk_composition_scores(model)
print(f"Q-composition shape: {result['q_composition'].shape}")
print(f"K-composition shape: {result['k_composition'].shape}")
print(f"\nTop Q-composition pairs:")
for sl, sh, dl, dh, score in result['top_q_pairs'][:5]:
    print(f"  L{sl}H{sh} -> L{dl}H{dh}: {score:.4f}")
print(f"\nTop K-composition pairs:")
for sl, sh, dl, dh, score in result['top_k_pairs'][:5]:
    print(f"  L{sl}H{sh} -> L{dl}H{dh}: {score:.4f}")

## V-Composition Scores

V-composition measures how earlier heads' outputs affect the values a later head
attends to, influencing what information gets passed through.

In [ ]:
result = v_composition_scores(model)
print(f"V-composition shape: {result['v_composition'].shape}")
print(f"Mean V-composition: {result['mean_v_composition']:.4f}")
print(f"\nTop V-composition pairs:")
for sl, sh, dl, dh, score in result['top_v_pairs'][:5]:
    print(f"  L{sl}H{sh} -> L{dl}H{dh}: {score:.4f}")

## Virtual Attention Patterns

When head B composes with head A, the combined effect creates a "virtual"
attention pattern: B's pattern composed with A's pattern.

In [ ]:
result = virtual_attention_patterns(model, tokens, src_layer=0, src_head=0, dst_layer=1, dst_head=0)
print(f"Source pattern shape: {result['src_pattern'].shape}")
print(f"Destination pattern shape: {result['dst_pattern'].shape}")
print(f"Virtual pattern shape: {result['virtual_pattern'].shape}")
print(f"Composition strength: {result['composition_strength']:.4f}")
print(f"\nVirtual pattern (last row):")
print(f"  {result['virtual_pattern'][-1]}")

## Composition Path Tracing

Trace chains of composing heads to find multi-step computation paths.
Identifies which head combinations matter most for a given metric.

In [ ]:
result = composition_path_tracing(model, tokens, metric_fn, max_depth=2)
print(f"Total paths evaluated: {len(result['path_scores'])}")
print(f"Significant paths: {result['n_significant_paths']}")
print(f"\nTop paths:")
for path, score in result['top_paths'][:10]:
    if len(path) == 2:
        print(f"  L{path[0]}H{path[1]}: {score:.4f}")
    elif len(path) == 4:
        print(f"  L{path[0]}H{path[1]} -> L{path[2]}H{path[3]}: {score:.4f}")

## Full Composition Matrix

Combine Q, K, and V composition into a single summary showing which heads
compose most strongly and what type of composition dominates.

In [ ]:
result = full_composition_matrix(model)
print(f"Total composition shape: {result['total_composition'].shape}")
sl, sh, dl, dh = result['most_composing_pair']
print(f"Most composing pair: L{sl}H{sh} -> L{dl}H{dh}")
print(f"\nLayer composition summary:")
for sl in range(cfg.n_layers):
    for dl in range(cfg.n_layers):
        val = result['layer_composition_summary'][sl, dl]
        if val > 0.01:
            print(f"  Layer {sl} -> Layer {dl}: {val:.4f}")
type_names = ['Q', 'K', 'V']
print(f"\nComposition type distribution:")
for t in range(3):
    frac = float(np.mean(result['composition_type'] == t))
    print(f"  {type_names[t]}-composition: {frac:.1%}")